### Week 4-5: Data Cleaning and Prepartion


In [2]:
import pandas as pd
import numpy as np


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
sold = pd.read_csv('data/sold_with_rates.csv')
listing = pd.read_csv('data/listings_with_rates.csv')


/var/folders/wj/4rpm0_z134d560x0595bhn900000gn/T/ipykernel_24566/2610127373.py:1: DtypeWarning: Columns (19,32) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('data/sold_with_rates.csv')


In [5]:
# Task: Converting data fields to datetime format

#print(sold.columns)
#print(sold.info())
#print(listing.info())

date_fields = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

for field in date_fields:
    sold[field] = pd.to_datetime(sold[field], errors = 'coerce')        #coerce helps malformed dates
    listing[field] = pd.to_datetime(listing[field], errors = 'coerce')

#print(sold.info())
#print(listing.info())

#things to note: listing has more null dates than sold



In [6]:
#Task: Remove unnecessary or redundant columns

#print(sold.columns)
#print(listing.columns)

#Ask team: Which columns to drop for further analysis? 
#Possible columns LotSize, CoListAgent, ListAgent names, SubdivisionName (it deosn't have that many rows). 


sold_null = pd.DataFrame(
    {'null_count': sold.isna().sum(), 
     'null_percent': ((sold.isna().sum()/len(sold)) * 100).round(2)}).sort_values('null_percent', ascending = False)

#print(sold_null[sold_null['null_count'] > 0])

listing_null = pd.DataFrame(
    {'null_count': listing.isna().sum(), 
     'null_percent': ((listing.isna().sum()/len(listing)) * 100).round(2)}).sort_values('null_percent', ascending = False)

#print(listing_null[listing_null['null_count'] > 0])


#redundant cols: lotsizes, listagentfirst and listagent last
#stateorprovince since we know its california based
#subdivisionname isn't relevant because we don't use it for further analysis

#complicated columns colistagents (too many null but may be usel), associationfee (understand HOA but many null), and keep the property variables 

cols_drop = ['StateOrProvince', 'SubdivisionName', 'ListAgentFirstName', 'ListAgentLastName']

#before counts: 
print(f"Sold before: {sold.shape}")
print(f"Listing before: {listing.shape}")

sold = sold.drop(columns = cols_drop)
listing = listing.drop(columns = cols_drop)

print(f"Sold after: {sold.shape}")
print(f"Listing after: {listing.shape}")


Sold before: (262311, 53)
Listing before: (400993, 50)
Sold after: (262311, 49)
Listing after: (400993, 46)


In [7]:
#Task: Handle missing values appropriately

#rows with few null values
#total rows in sold: (262311, 50), listing: (400993, 47)

#for columns that have missing value becuase there isn't anything to put so leave it as null
#for columns that could've been filled but weren't: flag them instead, don't compute mean
#for columnts that having a missing value but are important for analysis, so either flag or drop that row

sold_null = pd.DataFrame(
    {'null_count': sold.isna().sum(), 
     'null_percent': ((sold.isna().sum()/len(sold)) * 100).round(2)}).sort_values('null_percent', ascending = False)

#print(f"Sold: {sold_null[sold_null['null_count'] > 0]}")

listing_null = pd.DataFrame(
    {'null_count': listing.isna().sum(), 
     'null_percent': ((listing.isna().sum()/len(listing)) * 100).round(2)}).sort_values('null_percent', ascending = False)

#print(f"Listing: {listing_null[listing_null['null_count'] > 0]}")


#shape before
print(f"Sold before: {sold.shape}")
print(f"Listing before: {listing.shape}")

#you can make this more organized

#Sold
#dropping null values
sold = sold.dropna(subset=['ClosePrice'])

#livingarea, originallistprice will be used later
#flagging
sold['flag_living_area_null'] = sold['LivingArea'].isna()
sold['flag_original_list_price_null'] = sold['OriginalListPrice'].isna()

#filling in
#assuming that null = no HOA (property has no association fee)
sold['AssociationFee'] = sold['AssociationFee'].fillna(0)

#Listing
#flagging
listing['flag_living_area_null'] = listing['LivingArea'].isna()
listing['flag_original_list_price_null'] = listing['OriginalListPrice'].isna()

#filling in
#assuming that null = no HOA (property has no association fee)
listing['AssociationFee'] = listing['AssociationFee'].fillna(0)


#after shape
print(f"Sold after: {sold.shape}")
print(f"Listing after: {listing.shape}")

#debating in filling in associationfee




Sold before: (262311, 49)
Listing before: (400993, 46)
Sold after: (262310, 51)
Listing after: (400993, 48)


In [8]:
#Task: Ensure numeric fields are properly typed

#print(sold.head())
#print(f"sold:\n{sold.dtypes} \n")

#print(listing.head())
print(f"listing:\n{listing.dtypes}")

#this will help with time-series analysis
sold['year_month'] = sold['year_month'].astype('period[M]')
listing['year_month'] = listing['year_month'].astype('period[M]')

#print(sold['year_month'])
#print(listing['year_month'])

#shouldn't yearbuilt, bedroomstotal, bathroomtotalinteger be int64?

sold['BedroomsTotal'] = sold['BedroomsTotal'].astype('Int64')
sold['BathroomsTotalInteger'] = sold['BathroomsTotalInteger'].astype('Int64')
sold['YearBuilt'] = sold['YearBuilt'].astype('Int64')

listing['BedroomsTotal'] = listing['BedroomsTotal'].astype('Int64')
listing['BathroomsTotalInteger'] = listing['BathroomsTotalInteger'].astype('Int64')
listing['YearBuilt'] = listing['YearBuilt'].astype('Int64')


listing:
ListingKey                                int64
ListingId                                object
MlsStatus                                object
ClosePrice                              float64
ListPrice                               float64
OriginalListPrice                       float64
LivingArea                              float64
BedroomsTotal                           float64
BathroomsTotalInteger                   float64
LotSizeAcres                            float64
LotSizeArea                             float64
LotSizeSquareFeet                       float64
YearBuilt                               float64
Stories                                 float64
Levels                                   object
GarageSpaces                            float64
ParkingTotal                            float64
AttachedGarageYN                         object
FireplaceYN                              object
CloseDate                        datetime64[ns]
ListingContractDate            

In [9]:
#Task: Remove or flag invalid numeric values

#Best to flag invalid numeric values before removing it because then we reverse the decision


#SOLD
# dictionary allows for easier process of for loops
# the key is the flag column name and the value is a boolean condition
invalid_conditions_sold = {
    'flag_invalid_close_price': sold['ClosePrice'] <= 0,
    'flag_invalid_living_area': sold['LivingArea'] <= 0,
    'flag_negative_days_on_market': sold['DaysOnMarket'] < 0,
    'flag_negative_bedrooms': sold['BedroomsTotal'] < 0,
    'flag_negative_total_bathrooms': sold['BathroomsTotalInteger'] < 0,
    'flag_lot_size_area_null': sold['LotSizeArea'].isna()
}
#this is the for loop, and .items() helps go throguh key and value at the same time
for flag_name, condition in invalid_conditions_sold.items():
    sold[flag_name] = condition 
    print(f"{flag_name}: {condition.sum()}") #.sum() helps us get all the true values (following the boolean condition)



#LISTING

invalid_conditions_listing = {
    'flag_invalid_close_price': listing['ClosePrice'] <= 0,
    'flag_invalid_living_area': listing['LivingArea'] <= 0,
    'flag_negative_days_on_market': listing['DaysOnMarket'] < 0,
    'flag_negative_bedrooms': listing['BedroomsTotal'] < 0,
    'flag_negative_total_bathrooms': listing['BathroomsTotalInteger'] < 0,
    'flag_lot_size_area_null': sold['LotSizeArea'].isna()
}
#this is the for loop, and .items() helps go throguh key and value at the same time
for flag_name, condition in invalid_conditions_listing.items():
    listing[flag_name] = condition 
    print(f"{flag_name}: {condition.sum()}") #.sum() helps us get all the true values (following the boolean condition)


print(f"Sold Shape: {sold.shape}")
print(f"Listing Shape: {listing.shape}")


flag_invalid_close_price: 1
flag_invalid_living_area: 104
flag_negative_days_on_market: 22
flag_negative_bedrooms: 0
flag_negative_total_bathrooms: 0
flag_lot_size_area_null: 20202
flag_invalid_close_price: 0
flag_invalid_living_area: 262
flag_negative_days_on_market: 23
flag_negative_bedrooms: 0
flag_negative_total_bathrooms: 0
flag_lot_size_area_null: 20202
Sold Shape: (262310, 57)
Listing Shape: (400993, 54)


In [10]:
#Date Consistency Check

sold['listing_after_close_flag'] = ((sold['ListingContractDate'].notna()) & (sold['CloseDate'].notna()) &(sold['ListingContractDate'] > sold['CloseDate']))
sold['purchase_after_close_flag'] = ((sold['PurchaseContractDate'].notna()) & (sold['CloseDate'].notna()) & (sold['PurchaseContractDate'] > sold['CloseDate']))
sold['negative_timeline_flag'] = (
    sold['listing_after_close_flag'] | sold['purchase_after_close_flag'] |
    (sold['ListingContractDate'].notna() &
    sold['PurchaseContractDate'].notna() &
    (sold['PurchaseContractDate'] < sold['ListingContractDate'])
    )
)


listing['listing_after_close_flag'] = ((listing['ListingContractDate'].notna()) & (listing['CloseDate'].notna()) & (listing['ListingContractDate'] > listing['CloseDate']))
listing['purchase_after_close_flag'] = ((listing['PurchaseContractDate'].notna()) & (listing['CloseDate'].notna()) & (listing['PurchaseContractDate'] > listing['CloseDate']))
listing['negative_timeline_flag'] = (
    listing['listing_after_close_flag'] | listing['purchase_after_close_flag'] | 
    (listing['ListingContractDate'].notna() & 
     listing['PurchaseContractDate'].notna() & 
     (listing['PurchaseContractDate'] < listing['ListingContractDate'])
    )
)

print("Sold")
print(f"listing_after_close_flag: {sold['listing_after_close_flag'].sum()}")
print(f"purchase_after_close_flag: {sold['purchase_after_close_flag'].sum()}")
print(f"negative_timeline_flag: {sold['negative_timeline_flag'].sum()}")

print("\nListing")
print(f"listing_after_close_flag: {listing['listing_after_close_flag'].sum()}")
print(f"purchase_after_close_flag: {listing['purchase_after_close_flag'].sum()}")
print(f"negative_timeline_flag: {listing['negative_timeline_flag'].sum()} \n")


print(f"Sold Shape: {sold.shape}")
print(f"Listing Shape: {listing.shape}")

Sold
listing_after_close_flag: 41
purchase_after_close_flag: 154
negative_timeline_flag: 331

Listing
listing_after_close_flag: 55
purchase_after_close_flag: 219
negative_timeline_flag: 409 

Sold Shape: (262310, 60)
Listing Shape: (400993, 57)


In [12]:
#Geographic Data Checks


for df in [sold, listing]:
    df['geographic_missing_coords_flag'] = (df['Latitude'].isna() | df['Longitude'].isna())
    df['geographic_zero_flag'] = ((df['Latitude'] == 0) | (df['Longitude'] == 0))
    df['geographic_positive_longitude_flag'] = (df['Longitude'] > 0)
    df['geographic_out_of_state_flag'] = ((df['Latitude'] < 32.5) | (df['Latitude'] > 42) |  (df['Longitude'] < -125) | (df['Longitude'] > -114.1) )

columns = ['geographic_missing_coords_flag','geographic_zero_flag','geographic_positive_longitude_flag','geographic_out_of_state_flag']

print("Sold")
for col in columns:
    print(f"{col}: {sold[col].sum()}")

print("\nListing:")
for col in columns:
    print(f"{col}: {listing[col].sum()}")

print(f"\nSold Shape: {sold.shape}")
print(f"Listing Shape: {listing.shape}")

Sold
geographic_missing_coords_flag: 3639
geographic_zero_flag: 14
geographic_positive_longitude_flag: 25
geographic_out_of_state_flag: 63

Listing:
geographic_missing_coords_flag: 66188
geographic_zero_flag: 43
geographic_positive_longitude_flag: 48
geographic_out_of_state_flag: 219

Sold Shape: (262310, 64)
Listing Shape: (400993, 61)


In [13]:
#After tasks

#shape of data
print("Sold\n",sold.shape)
print("\nListing\n",listing.shape)

#any extra null values
print("\nSold:\n",sold.isnull().sum())
print("\nListing:\n",listing.isnull().sum())

#checking data types
print("\nSold:\n",sold.dtypes)
print("\nListing:\n",listing.dtypes)

#looking at values
print("\nSold:\n",sold.describe())
print("\nListing:\n",listing.describe())


Sold
 (262310, 64)

Listing
 (400993, 61)

Sold:
 ListingKey                                 0
ListingId                                  0
MlsStatus                                  0
ClosePrice                                 0
ListPrice                                  0
OriginalListPrice                        476
LivingArea                               157
BedroomsTotal                              7
BathroomsTotalInteger                     46
LotSizeAcres                           20532
LotSizeArea                            20202
LotSizeSquareFeet                      20286
YearBuilt                                249
Stories                                40433
Levels                                 25273
GarageSpaces                           11265
ParkingTotal                               2
AttachedGarageYN                       39921
PoolPrivateYN                          22746
FireplaceYN                              206
ViewYN                                 22683
Floor

In [16]:
sold['PostalCode'] = sold['PostalCode'].astype(str)
listing['PostalCode'] = listing['PostalCode'].astype(str)

sold.to_parquet('data/sold_prepared.parquet', index=False)
listing.to_parquet('data/listing_prepared.parquet', index=False)